# Attention Core Architecture

## 1. Imports

In [ ]:
# Cell 1 — Imports
import torch
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd
import json
import os
import torchvision.transforms as T
from torch.utils.data import Dataset, DataLoader, random_split
from PIL import Image
import torchvision.models as models

## 2. Reloading directories

In [ ]:
# Cell 2 — Reload df and vocabs
VOCAB_DIR = '/kaggle/input/datasets/manjushwarkhairkar/vocab_and_category'

df = pd.read_csv(os.path.join(VOCAB_DIR, 'fashion_with_indices.csv'))
df = df.dropna(subset=['proc_img_path', 'proc_sketch_path']).reset_index(drop=True)
print(f"Loaded {len(df)} rows")

with open(os.path.join(VOCAB_DIR, 'colour_vocab.json'))   as f:
    colour_vocab   = json.load(f)
with open(os.path.join(VOCAB_DIR, 'category_vocab.json')) as f:
    category_vocab = json.load(f)

print(f"Colour vocab size  : {len(colour_vocab)}")
print(f"Category vocab size: {len(category_vocab)}")

## 3. Dataset + Dataloader

In [ ]:
# Cell 3 — Rebuild dataset + dataloader
img_transform = T.Compose([
    T.Resize((224, 224)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406],
                std =[0.229, 0.224, 0.225])
])
sketch_transform = T.Compose([
    T.Resize((224, 224)),
    T.Grayscale(num_output_channels=1),
    T.ToTensor(),
    T.Normalize(mean=[0.5], std=[0.5])
])

class MyntraSketchDataset(Dataset):
    def __init__(self, dataframe, img_tfm=None, sketch_tfm=None):
        self.df         = dataframe.reset_index(drop=True)
        self.img_tfm    = img_tfm
        self.sketch_tfm = sketch_tfm

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row    = self.df.iloc[idx]
        img    = Image.open(row['proc_img_path']).convert('RGB')
        sketch = Image.open(row['proc_sketch_path']).convert('RGB')
        if self.img_tfm:    img    = self.img_tfm(img)
        if self.sketch_tfm: sketch = self.sketch_tfm(sketch)
        return {
            'sketch'      : sketch,
            'image'       : img,
            'name'        : row['name'],
            'colour'      : row['colour'],
            'colour_idx'  : torch.tensor(int(row['colour_idx']),   dtype=torch.long),
            'category_idx': torch.tensor(int(row['category_idx']), dtype=torch.long),
            'idx'         : idx
        }

full_dataset = MyntraSketchDataset(df, img_tfm=img_transform, sketch_tfm=sketch_transform)

total   = len(full_dataset)
n_train = int(0.8 * total)
n_val   = int(0.1 * total)
n_test  = total - n_train - n_val

train_ds, val_ds, test_ds = random_split(
    full_dataset,
    [n_train, n_val, n_test],
    generator=torch.Generator().manual_seed(42)
)

PIN          = torch.cuda.is_available()
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True,  num_workers=2, pin_memory=PIN)
val_loader   = DataLoader(val_ds,   batch_size=32, shuffle=False, num_workers=2, pin_memory=PIN)

print(f"Train: {len(train_ds)} | Val: {len(val_ds)} | Test: {len(test_ds)}")
print(f"Train batches: {len(train_loader)}")

## 4. Encoder, Embedder, LSTM fresh

In [ ]:
# Cell 4 — Rebuild encoder, embedder, lstm fresh
class SketchEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        resnet = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
        self.backbone      = nn.Sequential(*list(resnet.children())[:-2])
        self.input_adapter = nn.Conv2d(1, 3, kernel_size=1, bias=False)
        for param in self.backbone.parameters():
            param.requires_grad = False
    def forward(self, sketch):
        x = self.input_adapter(sketch)
        return self.backbone(x)           # [B, 512, 7, 7]

class StyleEmbedding(nn.Module):
    def __init__(self, colour_vocab_size, category_vocab_size, embed_dim=64):
        super().__init__()
        self.colour_emb   = nn.Embedding(colour_vocab_size,   embed_dim, padding_idx=0)
        self.category_emb = nn.Embedding(category_vocab_size, embed_dim, padding_idx=0)
        self.fusion = nn.Sequential(
            nn.Linear(embed_dim * 2, embed_dim * 2),
            nn.ReLU(),
            nn.Dropout(0.1)
        )
    def forward(self, colour_idx, category_idx):
        c     = self.colour_emb(colour_idx)
        s     = self.category_emb(category_idx)
        return self.fusion(torch.cat([c, s], dim=-1))  # [B, 128]

class LSTMCore(nn.Module):
    def __init__(self, encoder_dim=512, style_dim=128, hidden_size=256, num_layers=2, dropout=0.1):
        super().__init__()
        self.hidden_size = hidden_size
        self.num_layers  = num_layers
        self.encoder_proj = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(encoder_dim, hidden_size),
            nn.ReLU()
        )
        self.style_proj = nn.Sequential(
            nn.Linear(style_dim, hidden_size),
            nn.ReLU()
        )
        self.input_proj = nn.Sequential(
            nn.Linear(hidden_size * 2, hidden_size),
            nn.ReLU(),
            nn.Dropout(dropout)
        )
        self.gru = nn.GRU(
            input_size  = hidden_size,
            hidden_size = hidden_size,
            num_layers  = num_layers,
            batch_first = True,
            dropout     = dropout if num_layers > 1 else 0.0
        )
        self.layer_norm = nn.LayerNorm(hidden_size)

    def forward(self, encoder_features, style_vector, hidden=None):
        enc_ctx = self.encoder_proj(encoder_features)
        sty_ctx = self.style_proj(style_vector)
        fused   = self.input_proj(torch.cat([enc_ctx, sty_ctx], dim=-1))
        fused   = fused.unsqueeze(1)
        out, hidden = self.gru(fused, hidden)
        return self.layer_norm(out.squeeze(1)), hidden  # [B, 256]

encoder        = SketchEncoder()
style_embedder = StyleEmbedding(len(colour_vocab), len(category_vocab), embed_dim=64)
lstm_core      = LSTMCore(encoder_dim=512, style_dim=128, hidden_size=256, num_layers=2)

encoder.eval()
style_embedder.eval()
lstm_core.eval()
print("Encoder, StyleEmbedder, LSTMCore initialized!")

## 5. ConditionalSpatialAttention class

In [ ]:
# Cell 5 — ConditionalSpatialAttention class
class ConditionalSpatialAttention(nn.Module):

    def __init__(self,
                 encoder_dim  = 512,   # feature map channels
                 hidden_size  = 256,   # LSTM hidden size
                 style_dim    = 128,   # style vector size
                 attn_dim     = 256):  # internal attention dim
        super().__init__()

        # project condition (hidden + style) → encoder_dim
        # so it can interact with the feature map spatially
        self.condition_proj = nn.Sequential(
            nn.Linear(hidden_size + style_dim, encoder_dim),
            nn.ReLU()
        )

        # 2-layer conv to compute attention logits over spatial grid
        # input: feature map + broadcasted condition → encoder_dim * 2
        self.attn_conv = nn.Sequential(
            nn.Conv2d(encoder_dim * 2, attn_dim, kernel_size=1),
            nn.ReLU(),
            nn.Conv2d(attn_dim, 1, kernel_size=1)  # → [B, 1, 7, 7] raw logits
        )

    def forward(self, feature_map, hidden_state, style_vector, sketch=None):
        # feature_map  : [B, 512, 7, 7]
        # hidden_state : [B, 256]
        # style_vector : [B, 128]
        # sketch       : [B, 1, 224, 224] optional — used to build silhouette mask

        B, C, H, W = feature_map.shape

        # ── Step 1: build condition vector ──
        condition = torch.cat([hidden_state, style_vector], dim=-1)  # [B, 384]
        condition = self.condition_proj(condition)                    # [B, 512]

        # broadcast condition over spatial grid → [B, 512, 7, 7]
        condition = condition.unsqueeze(-1).unsqueeze(-1).expand(B, C, H, W)

        # ── Step 2: compute raw attention logits ──
        # concat feature map + condition along channel dim → [B, 1024, 7, 7]
        combined = torch.cat([feature_map, condition], dim=1)
        logits   = self.attn_conv(combined)                          # [B, 1, 7, 7]

        # ── Step 3: apply silhouette mask (anti-bleeding) ──
        if sketch is not None:
            # sketch: [B, 1, 224, 224] normalized to [-1, 1]
            # convert back to [0, 1] range for thresholding
            sketch_01 = (sketch * 0.5) + 0.5                        # [-1,1] → [0,1]

            # threshold: pixels > 0.1 = garment, else background
            silhouette = (sketch_01 > 0.1).float()                  # [B, 1, 224, 224]

            # downsample to match feature map spatial size [B, 1, 7, 7]
            mask = F.interpolate(silhouette, size=(H, W), mode='nearest')

            # set background locations to -inf before softmax
            logits = logits.masked_fill(mask == 0, float('-inf'))

        # ── Step 4: masked softmax → attention weights ──
        # flatten spatial dims for softmax then reshape back
        logits_flat  = logits.view(B, -1)                           # [B, 49]
        attn_weights = F.softmax(logits_flat, dim=-1)               # [B, 49] sums to 1
        attn_weights = attn_weights.view(B, 1, H, W)                # [B, 1, 7, 7]

        # ── Step 5: compute context vector ──
        # weighted sum over spatial locations
        context = (feature_map * attn_weights).sum(dim=[-2, -1])    # [B, 512]

        return context, attn_weights
        # context      → [B, 512]      feeds into decoder
        # attn_weights → [B, 1, 7, 7]  for visualization

## 6. Instantiate and Inspect

In [ ]:
# Cell 6 — Instantiate and inspect
attention = ConditionalSpatialAttention(
    encoder_dim = 512,
    hidden_size = 256,
    style_dim   = 128,
    attn_dim    = 256
)

print(attention)

total     = sum(p.numel() for p in attention.parameters())
trainable = sum(p.numel() for p in attention.parameters() if p.requires_grad)
print(f"\nTotal params    : {total:,}")
print(f"Trainable params: {trainable:,}   ← all trained from scratch")

## 7. Sanity check with real batch

In [ ]:
# Cell 7 — Sanity check with real batch
batch        = next(iter(train_loader))
sketches     = batch['sketch']            # [32, 1, 224, 224]
colour_idx   = batch['colour_idx']        # [32]
category_idx = batch['category_idx']      # [32]

with torch.no_grad():
    # full forward pass through all modules
    features     = encoder(sketches)                         # [32, 512, 7, 7]
    style_vec    = style_embedder(colour_idx, category_idx)  # [32, 128]
    lstm_out, _  = lstm_core(features, style_vec)            # [32, 256]

    # attention — WITH silhouette mask
    context, attn_weights = attention(
        feature_map  = features,
        hidden_state = lstm_out,
        style_vector = style_vec,
        sketch       = sketches       # ← sketch used to build mask
    )

print(f"Feature map shape    : {features.shape}")
print(f"LSTM output shape    : {lstm_out.shape}")
print(f"Attention weights    : {attn_weights.shape}")
print(f"Context vector shape : {context.shape}")
print(f"\nExpected context     : torch.Size([32, 512])")
print(f"Expected attn weights: torch.Size([32, 1, 7, 7])")
print(f"\nContext match        : {context.shape == torch.Size([32, 512])}")
print(f"Attn weights match   : {attn_weights.shape == torch.Size([32, 1, 7, 7])}")

# verify mask is working — background weights should be exactly 0
print(f"\nMin attention weight : {attn_weights.min().item():.6f}  ← should be 0.0 (masked)")
print(f"Max attention weight : {attn_weights.max().item():.6f}")
print(f"Attn weights sum     : {attn_weights.sum(dim=[-2,-1]).mean().item():.4f}  ← should be 1.0")

## 8. Visualize attention map on one sample

In [ ]:
# Cell 8 — Visualize attention map on one sample
import matplotlib.pyplot as plt
import numpy as np

sample_sketch  = sketches[0].squeeze().numpy()          # [224, 224]
sample_attn    = attn_weights[0].squeeze().detach().numpy()  # [7, 7]

# upsample attention to sketch size for overlay
attn_upsampled = F.interpolate(
    attn_weights[0].unsqueeze(0),
    size=(224, 224),
    mode='bilinear',
    align_corners=False
).squeeze().detach().numpy()

fig, axes = plt.subplots(1, 3, figsize=(12, 4))

axes[0].imshow(sample_sketch, cmap='gray')
axes[0].set_title('Input Sketch')
axes[0].axis('off')

axes[1].imshow(sample_attn, cmap='hot')
axes[1].set_title('Attention Map 7x7')
axes[1].axis('off')

axes[2].imshow(sample_sketch, cmap='gray', alpha=0.6)
axes[2].imshow(attn_upsampled, cmap='hot', alpha=0.5)
axes[2].set_title('Attention Overlay')
axes[2].axis('off')

plt.suptitle('Conditional Spatial Attention with Silhouette Mask', fontsize=13)
plt.tight_layout()
plt.savefig('/kaggle/working/attention_map.png', dpi=150, bbox_inches='tight')
plt.show()
print("Attention map saved!")

## 9. Saving the model in .pt format

In [ ]:
# Cell 9 — Save
torch.save(attention.state_dict(), '/kaggle/working/attention.pt')
print("Saved: attention.pt")